In [1]:
# ============================================================
# TASK V: Recursive Pointer-Based FP-Tree Growth Algorithm
# ============================================================


# ============================================================
# STEP 1: CREATE THE FP-TREE NODE
# ============================================================

class FPNode:

    def __init__(self, item, count, parent):

        # Name of the item stored in this node
        self.item = item

        # Number of transactions passing through this node
        self.count = count

        # Pointer to parent node
        self.parent = parent

        # Dictionary containing child nodes
        # Example: {"Milk": node_object}
        self.children = {}

        # Link to another node containing the same item
        self.node_link = None


# ============================================================
# STEP 2: CREATE THE FP-TREE
# ============================================================

class FPTree:

    def __init__(self, transactions, min_support):

        # Minimum support threshold
        self.min_support = min_support

        # Create root node
        # Root has no item and no parent
        self.root = FPNode(None, 1, None)

        # Header table
        # Format:
        # {
        #   item: [support_count, first_node]
        # }
        self.header_table = {}

        # Store original transactions
        self.transactions = transactions

        # Build the FP-Tree
        self.build_tree()


    # ========================================================
    # STEP 3: COUNT ITEM FREQUENCIES
    # ========================================================

    def build_tree(self):

        # Dictionary to count frequency of every item
        item_frequency = {}

        # Go through every transaction
        for transaction in self.transactions:

            # Go through every item
            for item in transaction:

                # Increase frequency
                item_frequency[item] = (
                    item_frequency.get(item, 0) + 1
                )


        # ====================================================
        # STEP 4: REMOVE INFREQUENT ITEMS
        # ====================================================

        # Keep only items satisfying minimum support
        frequent_items = {}

        for item, count in item_frequency.items():

            if count >= self.min_support:

                frequent_items[item] = count


        # If no frequent items exist
        if not frequent_items:

            return


        # ====================================================
        # STEP 5: CREATE HEADER TABLE
        # ====================================================

        for item, count in frequent_items.items():

            # Store:
            # [support_count, first_node_pointer]
            self.header_table[item] = [count, None]


        # ====================================================
        # STEP 6: SORT ITEMS BY FREQUENCY
        # ====================================================

        # Insert every transaction into tree
        for transaction in self.transactions:

            # Remove infrequent items
            filtered_transaction = [

                item for item in transaction
                if item in frequent_items

            ]

            # Sort according to frequency
            # Higher frequency comes first
            filtered_transaction.sort(

                key=lambda item: (
                    -frequent_items[item],
                    item
                )

            )


            # =================================================
            # STEP 7: INSERT TRANSACTION INTO TREE
            # =================================================

            if filtered_transaction:

                self.insert_transaction(

                    filtered_transaction,
                    self.root

                )


    # ========================================================
    # STEP 8: INSERT ITEMS INTO FP-TREE
    # ========================================================

    def insert_transaction(self, items, current_node):

        # Take first item
        first_item = items[0]


        # ----------------------------------------------------
        # CASE 1: CHILD ALREADY EXISTS
        # ----------------------------------------------------

        if first_item in current_node.children:

            # Get existing child
            child = current_node.children[first_item]

            # Increase count
            child.count += 1


        # ----------------------------------------------------
        # CASE 2: CREATE NEW NODE
        # ----------------------------------------------------

        else:

            # Create new FP node
            child = FPNode(

                first_item,
                1,
                current_node

            )

            # Add child to parent's children dictionary
            current_node.children[first_item] = child


            # ------------------------------------------------
            # ADD NODE TO HEADER TABLE LINK
            # ------------------------------------------------

            # Get first node of this item
            first_node = self.header_table[first_item][1]


            # If this is the first node
            if first_node is None:

                self.header_table[first_item][1] = child


            # Otherwise connect using node links
            else:

                current = first_node


                # Move to last linked node
                while current.node_link is not None:

                    current = current.node_link


                # Connect new node
                current.node_link = child


        # ----------------------------------------------------
        # RECURSIVELY INSERT REMAINING ITEMS
        # ----------------------------------------------------

        remaining_items = items[1:]


        if remaining_items:

            self.insert_transaction(

                remaining_items,
                child

            )


# ============================================================
# STEP 9: DISPLAY THE FP-TREE
# ============================================================

def display_tree(node, indent=0):

    # Do not print root item
    if node.item is not None:

        print(

            "  " * indent +

            f"{node.item} : {node.count}"

        )


    # Display all children
    for child in node.children.values():

        display_tree(

            child,
            indent + 1

        )


# ============================================================
# STEP 10: GET PREFIX PATH
# ============================================================

def get_prefix_path(node):

    # Store items in path
    path = []

    # Start from parent
    current = node.parent


    # Move upward until root
    while current is not None and current.item is not None:

        path.append(current.item)

        current = current.parent


    # Reverse path
    path.reverse()


    return path


# ============================================================
# STEP 11: CREATE CONDITIONAL PATTERN BASE
# ============================================================

def find_conditional_pattern_base(tree, item):

    # Store conditional paths
    conditional_patterns = []


    # Get first node of item
    node = tree.header_table[item][1]


    # Follow node links
    while node is not None:

        # Get prefix path using parent pointers
        prefix_path = get_prefix_path(node)


        # If prefix path exists
        if prefix_path:

            # Store path and count
            conditional_patterns.append(

                (
                    prefix_path,
                    node.count
                )

            )


        # Move to next node with same item
        node = node.node_link


    return conditional_patterns


# ============================================================
# STEP 12: BUILD CONDITIONAL FP-TREE
# ============================================================

def build_conditional_tree(

        conditional_patterns,
        min_support

):

    # Convert weighted paths into transactions
    transactions = []


    # For every path and count
    for path, count in conditional_patterns:

        # Repeat path according to count
        for _ in range(count):

            transactions.append(path)


    # Create conditional FP-Tree
    conditional_tree = FPTree(

        transactions,
        min_support

    )


    return conditional_tree


# ============================================================
# STEP 13: RECURSIVE FP-GROWTH ALGORITHM
# ============================================================

def fp_growth(

        tree,
        prefix,
        frequent_itemsets,
        min_support

):

    # Sort items by support
    items = sorted(

        tree.header_table.items(),

        key=lambda x: x[1][0]

    )


    # Process every item
    for item, data in items:

        # Support count
        support = data[0]


        # Create new frequent pattern
        new_pattern = prefix + [item]


        # Store pattern and support
        frequent_itemsets[

            tuple(sorted(new_pattern))

        ] = support


        # ----------------------------------------------------
        # FIND CONDITIONAL PATTERN BASE
        # ----------------------------------------------------

        conditional_patterns = (

            find_conditional_pattern_base(

                tree,
                item

            )

        )


        # ----------------------------------------------------
        # BUILD CONDITIONAL FP-TREE
        # ----------------------------------------------------

        conditional_tree = (

            build_conditional_tree(

                conditional_patterns,

                min_support

            )

        )


        # ----------------------------------------------------
        # RECURSIVE CALL
        # ----------------------------------------------------

        if conditional_tree.header_table:

            fp_growth(

                conditional_tree,

                new_pattern,

                frequent_itemsets,

                min_support

            )


# ============================================================
# STEP 14: GENERATE ASSOCIATION RULES
# ============================================================

def generate_association_rules(

        frequent_itemsets,
        min_confidence

):

    rules = []


    # Check every frequent itemset
    for itemset, support in frequent_itemsets.items():

        # Rules require at least 2 items
        if len(itemset) < 2:

            continue


        # Generate possible left-hand sides
        from itertools import combinations


        for r in range(

            1,

            len(itemset)

        ):


            # Create combinations
            for antecedent in combinations(

                    itemset,

                    r

            ):


                # Convert antecedent to tuple
                antecedent = tuple(

                    sorted(antecedent)

                )


                # Consequent =
                # Items not present in antecedent
                consequent = tuple(

                    item

                    for item in itemset

                    if item not in antecedent

                )


                # Get support of antecedent
                if antecedent in frequent_itemsets:

                    antecedent_support = (

                        frequent_itemsets[antecedent]

                    )


                    # Calculate confidence
                    confidence = (

                        support /

                        antecedent_support

                    )


                    # Keep strong rules
                    if confidence >= min_confidence:

                        rules.append(

                            (

                                antecedent,

                                consequent,

                                support,

                                confidence

                            )

                        )


    return rules


# ============================================================
# STEP 15: INPUT TRANSACTION DATA
# ============================================================

transactions = [

    ["Bread", "Milk", "Eggs"],

    ["Bread", "Milk"],

    ["Bread", "Eggs"],

    ["Milk", "Eggs"],

    ["Bread", "Milk", "Eggs"],

    ["Bread", "Milk"]

]


# ============================================================
# STEP 16: SET MINIMUM SUPPORT
# ============================================================

min_support = 2


# ============================================================
# STEP 17: BUILD MAIN FP-TREE
# ============================================================

tree = FPTree(

    transactions,

    min_support

)


# ============================================================
# STEP 18: DISPLAY FP-TREE
# ============================================================

print("\nFP-TREE")

print("-" * 40)


display_tree(tree.root)


# ============================================================
# STEP 19: DISPLAY HEADER TABLE
# ============================================================

print("\nHEADER TABLE")

print("-" * 40)


for item, data in tree.header_table.items():

    print(

        item,

        "-> Support:",

        data[0]

    )


# ============================================================
# STEP 20: MINE FREQUENT ITEMSETS
# ============================================================

frequent_itemsets = {}


fp_growth(

    tree,

    [],

    frequent_itemsets,

    min_support

)


# ============================================================
# STEP 21: DISPLAY FREQUENT ITEMSETS
# ============================================================

print("\nFREQUENT ITEMSETS")

print("-" * 40)


for itemset, support in sorted(

        frequent_itemsets.items(),

        key=lambda x: (

            len(x[0]),

            x[0]

        )

):

    print(

        itemset,

        "-> Support:",

        support

    )


# ============================================================
# STEP 22: GENERATE ASSOCIATION RULES
# ============================================================

min_confidence = 0.60


rules = generate_association_rules(

    frequent_itemsets,

    min_confidence

)


# ============================================================
# STEP 23: DISPLAY ASSOCIATION RULES
# ============================================================

print("\nASSOCIATION RULES")

print("-" * 40)


for antecedent, consequent, support, confidence in rules:

    print(

        antecedent,

        "->",

        consequent,

        "| Support:",

        support,

        "| Confidence:",

        round(

            confidence * 100,

            2

        ),

        "%"

    )


FP-TREE
----------------------------------------
  Bread : 5
    Milk : 4
      Eggs : 2
    Eggs : 1
  Milk : 1
    Eggs : 1

HEADER TABLE
----------------------------------------
Bread -> Support: 5
Milk -> Support: 5
Eggs -> Support: 4

FREQUENT ITEMSETS
----------------------------------------
('Bread',) -> Support: 5
('Eggs',) -> Support: 4
('Milk',) -> Support: 5
('Bread', 'Eggs') -> Support: 3
('Bread', 'Milk') -> Support: 4
('Eggs', 'Milk') -> Support: 3
('Bread', 'Eggs', 'Milk') -> Support: 2

ASSOCIATION RULES
----------------------------------------
('Bread',) -> ('Eggs',) | Support: 3 | Confidence: 60.0 %
('Eggs',) -> ('Bread',) | Support: 3 | Confidence: 75.0 %
('Eggs',) -> ('Milk',) | Support: 3 | Confidence: 75.0 %
('Milk',) -> ('Eggs',) | Support: 3 | Confidence: 60.0 %
('Bread', 'Eggs') -> ('Milk',) | Support: 2 | Confidence: 66.67 %
('Eggs', 'Milk') -> ('Bread',) | Support: 2 | Confidence: 66.67 %
('Bread',) -> ('Milk',) | Support: 4 | Confidence: 80.0 %
('Milk',) ->